# Machine Learning Model Development

- Training models for a supervised learning

- Compute and visualize results

- Evaluation of metric with MLFlow

- [Dataset "Discover São Paulo: Apartment Prices Insights $$"
](https://www.kaggle.com/datasets/marcelobatalhah/discover-so-paulo-apartment-prices-insights?utm_source=chatgpt.com): 
  - `created_date`: Date when the listing was created.
  - `Price`: The listed price of the apartment.
  - `below_price`: A boolean value indicating whether the property is below the average price (true or false).
  - `Area`: Apartment size in square meters.
  - `Address`: Full address, including street name, number, neighborhood, and city.
  - `Bedrooms`: Number of bedrooms.
  - `Bathrooms`: Number of bathrooms.
  - `Parking_Spaces`: Number of parking spaces.
  - `extract_date`: Date when the data was scraped.

## Env setup

In [0]:
%run ./env_config

In [0]:
EXPORT_AS_TABLE = True

TABLE_NAME = r'sp_ap_rents'

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import mlflow
from mlflow.models.signature import infer_signature

from utils.eda import check_missing_values
from utils.models import compute_metrics

## Prepare data

In [0]:
if EXPORT_AS_TABLE:
    data_path = '/Volumes/studies/ml_model_dev/data_bronze/SaoPaulo_OnlyAppartments.csv'
    df = spark.read.csv(data_path, header=True, inferSchema=True, sep=',')
    df = df.select([F.col(c).alias(c.lower()) for c in df.columns])
    df = df.withColumn('price', F.col('price').cast('double'))

    display(df.limit(5))

    df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TABLE_NAME)

else:
    df = spark.read.table(TABLE_NAME)
    print(f'#rows: {df.count()}')
    display(df.limit(5))

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date
2742871875,2018-05-24T15:02:18.000Z,2000000.0,false,62,"Alameda Casa Branca 909, Jardim Paulista - São Paulo/SP",1,1,1,2024-11-22
2670612721,2018-12-18T14:40:10.000Z,1015680.0,false,42,"Rua Doutor Guilherme Bannitz 61, Itaim Bibi - São Paulo/SP",1,1,1,2024-11-22
2752273106,2018-03-28T09:37:42.000Z,490000.0,false,31,"Avenida Professor Francisco Morato 292, Butantã - São Paulo/SP",1,1,1,2024-11-22
2730614854,2022-03-17T20:07:18.000Z,850000.0,false,50,"Avenida Doutor Cardoso de Melo 04545003, Vila Olímpia - São Paulo/SP",1,1,1,2024-11-22
2756989746,2018-03-27T23:24:33.000Z,209100.0,false,37,"Rua Serra de São Domingos 100, Vila Carmosina - São Paulo/SP",1,1,1,2024-11-22


In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- below_price: boolean (nullable = true)
 |-- area: integer (nullable = true)
 |-- adress: string (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: integer (nullable = true)
 |-- parking_spaces: integer (nullable = true)
 |-- extract_date: date (nullable = true)



In [0]:
check_missing_values(df)

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date
0,0,0,0,0,0,0,0,0,0


In [0]:
df = df \
       .withColumn('price_per_m2', F.col('Price') / F.col('Area')) \
       .withColumn('year', F.year('created_date')) \
       .withColumn('month', F.month('created_date')) \
       .withColumn('neighbourhood', F.split(F.col('adress'), ', ')) \
       .withColumn('street', 
              F.when(F.size(F.col('neighbourhood')) > 0, F.col('neighbourhood').getItem(0))
              .otherwise(F.lit(None))) \
       .withColumn('neighbourhood_city', 
              F.when(F.size(F.col('neighbourhood')) > 1, F.col('neighbourhood').getItem(1))
              .otherwise(F.lit(None))) \
       .withColumn('neighbourhood', F.split(F.col('neighbourhood_city'), ' - ')[0]) \
       .withColumn('month_sin', F.sin(2 * F.pi() * F.col('month') / 12)) \
       .withColumn('month_cos', F.cos(2 * F.pi() * F.col('month') / 12)) \
       .drop('neighbourhood_city')

display(df.limit(3))

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date,price_per_m2,year,month,neighbourhood,street,month_sin,month_cos
2742871875,2018-05-24T15:02:18.000Z,2000000.0,false,62,"Alameda Casa Branca 909, Jardim Paulista - São Paulo/SP",1,1,1,2024-11-22,32258.064516129034,2018,5,Jardim Paulista,Alameda Casa Branca 909,0.49999999999999994,-0.8660254037844387
2670612721,2018-12-18T14:40:10.000Z,1015680.0,false,42,"Rua Doutor Guilherme Bannitz 61, Itaim Bibi - São Paulo/SP",1,1,1,2024-11-22,24182.85714285714,2018,12,Itaim Bibi,Rua Doutor Guilherme Bannitz 61,-2.4492935982947064E-16,1.0
2752273106,2018-03-28T09:37:42.000Z,490000.0,false,31,"Avenida Professor Francisco Morato 292, Butantã - São Paulo/SP",1,1,1,2024-11-22,15806.451612903225,2018,3,Butantã,Avenida Professor Francisco Morato 292,1.0,6.123233995736766E-17


In [0]:
check_missing_values(df)

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date,price_per_m2,year,month,neighbourhood,street,month_sin,month_cos
0,0,0,0,0,0,0,0,0,0,0,0,0,1776,0,0,0


In [0]:
df = df.dropna()

check_missing_values(df)

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date,price_per_m2,year,month,neighbourhood,street,month_sin,month_cos
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Features pipeline

In [0]:
num_cols = [
    'area',
    'price_per_m2',
    'bedrooms',
    'bathrooms',
    'parking_Spaces',
    'year',
    'month_sin',
    'month_cos'
]

cat_cols = [
    'neighbourhood_index'
]

indexer = StringIndexer(
    inputCol='neighbourhood',
    outputCol='neighbourhood_index',
    handleInvalid='keep'
)

assembler = VectorAssembler(
    inputCols=num_cols + cat_cols,
    outputCol='features_raw'
)

scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True
)

features_pipeline = Pipeline(stages=[
    indexer,
    assembler,
    scaler
])

print(f'Stages features pipeline: {features_pipeline.getStages()}')

Stages features pipeline: [StringIndexer_f26eb09194ec, VectorAssembler_1a26b1526e66, StandardScaler_ea916d635506]


In [0]:
with mlflow.start_run(run_name='ML features tracking') as run:
    model_path = '/Volumes/studies/ml_model_dev/data_bronze/experiment_artifacts'
    model_name = 'studies.ml_model_dev.features_model'

    features_model = features_pipeline.fit(df)
    df_prep = features_model.transform(df)

    mlflow.log_input(mlflow.data.from_spark(df), context='input')

    signature = infer_signature(df, df_prep)
   
    mlflow.spark.log_model(
        spark_model=features_model,
        artifact_path='artifacts-features-model',
        signature=signature,
        dfs_tmpdir=model_path,
        registered_model_name=model_name
    )

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/03 12:52:03 WARNING mlflow.data.spark_dataset: Encountered an unexpected exception while computing Spark dataset profile. Exception: [NOT_IMPLEMENTED] rdd is not implemented.
/databricks/python/lib/python3.12/site-packages/mlflow/ty

Uploading artifacts:   0%|          | 0/32 [00:00<?, ?it/s]

🔗 Created version '2' of model 'studies.ml_model_dev.features_model': https://dbc-34ac7b3c-7a54.cloud.databricks.com/explore/data/models/studies/ml_model_dev/features_model/version/2?o=7474656785376246


In [0]:
display(df_prep.limit(3))

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date,price_per_m2,year,month,neighbourhood,street,month_sin,month_cos,neighbourhood_index,features_raw,features
2742871875,2018-05-24T15:02:18.000Z,2000000.0,false,62,"Alameda Casa Branca 909, Jardim Paulista - São Paulo/SP",1,1,1,2024-11-22,32258.064516129034,2018,5,Jardim Paulista,Alameda Casa Branca 909,0.49999999999999994,-0.8660254037844387,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""62.0"",""32258.064516129034"",""1.0"",""1.0"",""1.0"",""2018.0"",""0.49999999999999994"",""-0.8660254037844387"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-0.8079419541803585"",""1.405240234150604"",""-1.892841351748665"",""-1.2929911318672702"",""-0.3852689905821582"",""-0.7037276106977938"",""0.008773608973931606"",""-1.8412520914060075"",""-0.6331273709380327""]}"
2670612721,2018-12-18T14:40:10.000Z,1015680.0,false,42,"Rua Doutor Guilherme Bannitz 61, Itaim Bibi - São Paulo/SP",1,1,1,2024-11-22,24182.85714285714,2018,12,Itaim Bibi,Rua Doutor Guilherme Bannitz 61,-2.4492935982947064E-16,1.0,5.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""42.0"",""24182.85714285714"",""1.0"",""1.0"",""1.0"",""2018.0"",""-2.4492935982947064E-16"",""1.0"",""5.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.025690315290806"",""0.7929832438640954"",""-1.892841351748665"",""-1.2929911318672702"",""-0.3852689905821582"",""-0.7037276106977938"",""-0.6625783357768487"",""2.330924732962641"",""-0.582309050883667""]}"
2752273106,2018-03-28T09:37:42.000Z,490000.0,false,31,"Avenida Professor Francisco Morato 292, Butantã - São Paulo/SP",1,1,1,2024-11-22,15806.451612903225,2018,3,Butantã,Avenida Professor Francisco Morato 292,1.0,6.123233995736766E-17,82.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""31.0"",""15806.451612903225"",""1.0"",""1.0"",""1.0"",""2018.0"",""1.0"",""6.123233995736766E-17"",""82.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""-1.145451913901552"",""0.1578896045151291"",""-1.892841351748665"",""-1.2929911318672702"",""-0.3852689905821582"",""-0.7037276106977938"",""0.6801255537247117"",""0.09506191142379022"",""0.2002930779535642""]}"


## Training the model

### Split into training and test sets

In [0]:
df_train, df_test = df_prep.randomSplit([0.8, 0.2], seed=42)

print(f'#rows df_train: {df_train.count()}')
print(f'#rows df_test: {df_test.count()}')

#rows df_train: 20973
#rows df_test: 5079


In [0]:
display(df_train.limit(1))

id,created_date,price,below_price,area,adress,bedrooms,bathrooms,parking_spaces,extract_date,price_per_m2,year,month,neighbourhood,street,month_sin,month_cos,neighbourhood_index,features_raw,features
44326867,2018-03-27T17:06:14.000Z,3200000.0,true,288,"Rua Alagoas 710, Higienópolis - São Paulo/SP",3,1,4,2024-11-22,11111.111111111111,2018,3,Higienópolis,Rua Alagoas 710,1.0,6.123233995736766E-17,23.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""288.0"",""11111.111111111111"",""3.0"",""1.0"",""4.0"",""2018.0"",""1.0"",""6.123233995736766E-17"",""23.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.6526145263676977"",""-0.19810806973813277"",""0.2789327562296816"",""-1.2929911318672702"",""0.6110939103125517"",""-0.7037276106977938"",""0.6801255537247117"",""0.09506191142379022"",""-0.39936309868795067""]}"


In [0]:
with mlflow.start_run(run_name='ML experiment tracking') as run:
    model_name = 'studies.ml_model_dev.regression_model'
    model_path = '/Volumes/studies/ml_model_dev/data_bronze/linear_regression'
    
    lr = LinearRegression(
        featuresCol='features',
        labelCol='price'
    )
    
    model_lr = lr.fit(df_train)
    pred_train = model_lr.transform(df_train)
    pred_test = model_lr.transform(df_test)

    mlflow.log_input(mlflow.data.from_spark(df_train), context='training_data')
    mlflow.log_input(mlflow.data.from_spark(df_test), context='test_data')

    signature = infer_signature(df_train, pred_train)
    
    mlflow.spark.log_model(
        spark_model=model_lr,
        artifact_path='artifacts-regression-model',
        signature=signature,
        dfs_tmpdir=model_path,
        registered_model_name=model_name
    )

    pred_train_pd = pred_train.select('price', 'prediction').toPandas()
    pred_test_pd = pred_test.select('price', 'prediction').toPandas()
    train_metrics = compute_metrics(y=pred_train_pd['price'], y_pred=pred_train_pd['prediction'], suffix='train')
    test_metrics = compute_metrics(y=pred_test_pd['price'], y_pred=pred_test_pd['prediction'], suffix='test')

    all_metrics = pd.concat([train_metrics, test_metrics], axis=1).T
    display(all_metrics)

    mlflow.log_metrics(train_metrics.iloc[0].to_dict())
    mlflow.log_metrics(test_metrics.iloc[0].to_dict())


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/03 12:52:39 WARNING mlflow.data.spark_dataset: Encountered an unexpected exception while computing Spark dataset profile. Exception: [NOT_IMPLEMENTED] rdd is not implemented.
/databricks/python/lib/python3.12/site-packages/mlflow/ty

Uploading artifacts:   0%|          | 0/20 [00:00<?, ?it/s]

🔗 Created version '5' of model 'studies.ml_model_dev.regression_model': https://dbc-34ac7b3c-7a54.cloud.databricks.com/explore/data/models/studies/ml_model_dev/regression_model/version/5?o=7474656785376246


metrics
0.7576204141875891
510373.65888846386
1.9650964442482136E12
0.38547893255646437
255625.43575072172
5.788575705682543E7
0.7576204141875891
0.7626662421302393
523962.52969624277
2.4566178808506074E12


<img src='./imgs/mlflow_artifacts.png'>